# Plot the QuantUI results from the GPU session

The GPU timing sweep and optional CPU geometry calculation write two kinds of results to
`$COURSE_WORK/quantui`:

1. **CPU vs GPU wall times** (`run_cpu_gpu_comparison.py`) &rarr; a bar chart, and
2. **a geometry-relaxation trajectory** (`run_geometry_optimization.py`) &rarr; a line plot.

This notebook is deliberately thin: every plotting decision lives in
[`quantui_result_plots.py`](quantui_result_plots.py) next to it, so the logic is
readable and version-controlled rather than buried in notebook cells. Here we
only import and call.

Each plot falls back independently when its own result is absent, so real timing
results can be shown alongside a clearly watermarked synthetic trajectory.

In [ ]:
%matplotlib inline
import sys, pathlib
# Find the sidecar whether Jupyter started in this directory or the repository root.
start = pathlib.Path.cwd().resolve()
for candidate in (start, *start.parents):
    lesson_dir = candidate / 'tutorials' / '05-visualization-postprocessing'
    if (lesson_dir / 'quantui_result_plots.py').exists():
        sys.path.insert(0, str(lesson_dir))
        break
else:
    raise FileNotFoundError('Could not locate quantui_result_plots.py')

from quantui_result_plots import (
    find_result_source,
    find_trajectory_source,
    load_comparison_results,
    plot_compute_time_bars,
    load_trajectory,
    plot_relaxation_trajectory,
    save_figure,
)

source, is_sample = find_result_source()
trajectory_source, trajectory_is_sample = find_trajectory_source()
print("Timing source:", source)
print("  -> bundled SAMPLE data" if is_sample else "  -> your own GPU-session results")
print("Trajectory source:", trajectory_source)
print("  -> bundled SYNTHETIC data" if trajectory_is_sample else "  -> your own CPU result")

## 1. CPU vs GPU wall time — where does the GPU start to win?

Grouped bars, one pair per system. Watch the GPU bars stay nearly flat while the
CPU bar for the largest basis set towers: that flat stretch is fixed launch and
transfer overhead, and the crossover is the system where the two bars are level.

In [ ]:
results = load_comparison_results(source)
fig, ax = plot_compute_time_bars(results, is_sample=is_sample)
save_figure(fig, "cpu_gpu_walltime")

## 2. Geometry relaxation — energy vs optimization step

A geometry optimization moves the atoms downhill in energy until the forces
fall below a chosen threshold. The line falls steeply at first, then levels
off near the final recorded structure. (This is a CPU calculation — it does
not need a GPU.)

In [ ]:
traj = load_trajectory(trajectory_source, preset="water")
if traj is None:
    print("No geometry_optimization_water.json found — run run_geometry_optimization.py first.")
else:
    fig, ax = plot_relaxation_trajectory(traj)
    save_figure(fig, "geometry_relaxation")

When `$COURSE_WORK` is set, both figures are saved under its `visualization/`
directory as **PNG** (for slides) and **PDF** (vector, for print). Otherwise
they are saved in Jupyter's current directory. Before using one in a report, run it past the
figure checklist in the [session README](README.md): does it answer one
question, are the axes and units labelled, and — for the timings — is the CPU
allocation named, since a speedup without its denominator is not a result?